In [1]:
!pip -q install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.7 MB/s eta 0:00:00


In [2]:
import os
import math
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

from torch_geometric.data import HeteroData
from torch_geometric.transforms import ToUndirected
from torch_geometric.nn import HeteroConv, SAGEConv

In [4]:
ARTIFACT_DIR = Path("/content/pharmgraph_pass1_artifacts")
OUT_DIR = ARTIFACT_DIR / "pass2_hetero_gnn"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_GRAPH_FILE = ARTIFACT_DIR / "train_graph_edges.csv"

TRAIN_POS_FILE = ARTIFACT_DIR / "gene_drug_train_pos.csv"
VAL_POS_FILE   = ARTIFACT_DIR / "gene_drug_val_pos.csv"
TEST_POS_FILE  = ARTIFACT_DIR / "gene_drug_test_pos.csv"

TRAIN_NEG_FILE = ARTIFACT_DIR / "gene_drug_train_neg.csv"
VAL_NEG_FILE   = ARTIFACT_DIR / "gene_drug_val_neg.csv"
TEST_NEG_FILE  = ARTIFACT_DIR / "gene_drug_test_neg.csv"

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [5]:
train_graph_edges = pd.read_csv(TRAIN_GRAPH_FILE)

train_pos = pd.read_csv(TRAIN_POS_FILE)
val_pos   = pd.read_csv(VAL_POS_FILE)
test_pos  = pd.read_csv(TEST_POS_FILE)

train_neg = pd.read_csv(TRAIN_NEG_FILE)
val_neg   = pd.read_csv(VAL_NEG_FILE)
test_neg  = pd.read_csv(TEST_NEG_FILE)

print("train_graph_edges:", train_graph_edges.shape)
print("train_pos:", train_pos.shape)
print("val_pos:", val_pos.shape)
print("test_pos:", test_pos.shape)
print("train_neg:", train_neg.shape)
print("val_neg:", val_neg.shape)
print("test_neg:", test_neg.shape)

display(train_graph_edges.head())

train_graph_edges: (68122, 6)
train_pos: (51535, 5)
val_pos: (8629, 5)
test_pos: (8842, 5)
train_neg: (51535, 3)
val_neg: (8629, 3)
test_neg: (8842, 3)


,relation_type,source_id,target_id,source_type,target_type,split
0,gene_disease,gene:nqo1,disease:pa151958383,gene,disease,NaN
1,gene_disease,gene:nqo1,disease:pa164925725,gene,disease,NaN
2,gene_disease,gene:nqo1,disease:pa166122058,gene,disease,NaN
3,gene_disease,gene:nqo1,disease:pa166123026,gene,disease,NaN
4,gene_disease,gene:nqo1,disease:pa166123431,gene,disease,NaN


In [6]:
def infer_node_type(node_id: str) -> str:
    return str(node_id).split(":", 1)[0]

all_nodes = set(train_graph_edges["source_id"]).union(set(train_graph_edges["target_id"]))

for df in [train_pos, val_pos, test_pos, train_neg, val_neg, test_neg]:
    all_nodes.update(df["source_id"].tolist())
    all_nodes.update(df["target_id"].tolist())

node_types = ["gene", "drug", "variant", "disease"]

nodes_by_type = {nt: [] for nt in node_types}
for nid in sorted(all_nodes):
    nt = infer_node_type(nid)
    if nt in nodes_by_type:
        nodes_by_type[nt].append(nid)

node2idx = {
    nt: {nid: i for i, nid in enumerate(nodes)}
    for nt, nodes in nodes_by_type.items()
}

for nt in node_types:
    print(nt, len(nodes_by_type[nt]))

gene 5292
drug 18481
variant 2826
disease 756


In [7]:
data = HeteroData()

for nt in node_types:
    data[nt].num_nodes = len(nodes_by_type[nt])

edge_triplets = []
for _, row in train_graph_edges.iterrows():
    src_id = row["source_id"]
    dst_id = row["target_id"]
    rel = row["relation_type"]

    src_type = infer_node_type(src_id)
    dst_type = infer_node_type(dst_id)

    if src_type not in node2idx or dst_type not in node2idx:
        continue

    edge_triplets.append((src_type, rel, dst_type, src_id, dst_id))

from collections import defaultdict
edge_dict = defaultdict(lambda: [[], []])

for src_type, rel, dst_type, src_id, dst_id in edge_triplets:
    edge_dict[(src_type, rel, dst_type)][0].append(node2idx[src_type][src_id])
    edge_dict[(src_type, rel, dst_type)][1].append(node2idx[dst_type][dst_id])

for edge_type, (src_idx, dst_idx) in edge_dict.items():
    edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)
    data[edge_type].edge_index = edge_index

data = ToUndirected()(data)

print(data)
print("node types:", data.node_types)
print("edge types:", data.edge_types)

HeteroData(
  gene={ num_nodes=5292 },
  drug={ num_nodes=18481 },
  variant={ num_nodes=2826 },
  disease={ num_nodes=756 },
  (gene, gene_disease, disease)={ edge_index=[2, 9425] },
  (variant, variant_drug, drug)={ edge_index=[2, 4332] },
  (variant, variant_gene, gene)={ edge_index=[2, 2830] },
  (gene, gene_drug, drug)={ edge_index=[2, 51535] },
  (disease, rev_gene_disease, gene)={ edge_index=[2, 9425] },
  (drug, rev_variant_drug, variant)={ edge_index=[2, 4332] },
  (gene, rev_variant_gene, variant)={ edge_index=[2, 2830] },
  (drug, rev_gene_drug, gene)={ edge_index=[2, 51535] }
)
node types: ['gene', 'drug', 'variant', 'disease']
edge types: [('gene', 'gene_disease', 'disease'), ('variant', 'variant_drug', 'drug'), ('variant', 'variant_gene', 'gene'), ('gene', 'gene_drug', 'drug'), ('disease', 'rev_gene_disease', 'gene'), ('drug', 'rev_variant_drug', 'variant'), ('gene', 'rev_variant_gene', 'variant'), ('drug', 'rev_gene_drug', 'gene')]


In [8]:
def build_edge_label_tensors(pos_df, neg_df):
    pos_src = [node2idx["gene"][x] for x in pos_df["source_id"]]
    pos_dst = [node2idx["drug"][x] for x in pos_df["target_id"]]

    neg_src = [node2idx["gene"][x] for x in neg_df["source_id"]]
    neg_dst = [node2idx["drug"][x] for x in neg_df["target_id"]]

    src = torch.tensor(pos_src + neg_src, dtype=torch.long)
    dst = torch.tensor(pos_dst + neg_dst, dtype=torch.long)

    edge_label_index = torch.stack([src, dst], dim=0)
    edge_label = torch.tensor([1] * len(pos_src) + [0] * len(neg_src), dtype=torch.float)

    return edge_label_index, edge_label

train_edge_label_index, train_edge_label = build_edge_label_tensors(train_pos, train_neg)
val_edge_label_index, val_edge_label = build_edge_label_tensors(val_pos, val_neg)
test_edge_label_index, test_edge_label = build_edge_label_tensors(test_pos, test_neg)

print(train_edge_label_index.shape, train_edge_label.shape)
print(val_edge_label_index.shape, val_edge_label.shape)
print(test_edge_label_index.shape, test_edge_label.shape)

torch.Size([2, 103070]) torch.Size([103070])
torch.Size([2, 17258]) torch.Size([17258])
torch.Size([2, 17684]) torch.Size([17684])


In [9]:
class EdgeDecoder(nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_channels, 1)
        )

    def forward(self, z_src, z_dst, edge_label_index):
        src, dst = edge_label_index
        h = torch.cat([z_src[src], z_dst[dst]], dim=-1)
        return self.mlp(h).view(-1)


class HeteroLinkPredictor(nn.Module):
    def __init__(self, metadata, num_nodes_dict, hidden_channels=64, out_channels=64):
        super().__init__()

        self.embeddings = nn.ModuleDict({
            node_type: nn.Embedding(num_nodes_dict[node_type], hidden_channels)
            for node_type in metadata[0]
        })

        conv1_dict = {}
        conv2_dict = {}
        for edge_type in metadata[1]:
            conv1_dict[edge_type] = SAGEConv((-1, -1), hidden_channels)
            conv2_dict[edge_type] = SAGEConv((-1, -1), out_channels)

        self.conv1 = HeteroConv(conv1_dict, aggr="sum")
        self.conv2 = HeteroConv(conv2_dict, aggr="sum")

        self.decoder = EdgeDecoder(out_channels)

    def forward(self, data):
        x_dict = {
            node_type: self.embeddings[node_type].weight
            for node_type in data.node_types
        }

        x_dict = self.conv1(x_dict, data.edge_index_dict)
        x_dict = {k: F.relu(v) for k, v in x_dict.items()}
        x_dict = {k: F.dropout(v, p=0.2, training=self.training) for k, v in x_dict.items()}

        x_dict = self.conv2(x_dict, data.edge_index_dict)

        return x_dict

    def decode(self, z_dict, edge_label_index):
        return self.decoder(z_dict["gene"], z_dict["drug"], edge_label_index)

In [10]:
num_nodes_dict = {nt: data[nt].num_nodes for nt in data.node_types}

model = HeteroLinkPredictor(
    metadata=data.metadata(),
    num_nodes_dict=num_nodes_dict,
    hidden_channels=64,
    out_channels=64
).to(device)

data = data.to(device)
train_edge_label_index = train_edge_label_index.to(device)
train_edge_label = train_edge_label.to(device)
val_edge_label_index = val_edge_label_index.to(device)
val_edge_label = val_edge_label.to(device)
test_edge_label_index = test_edge_label_index.to(device)
test_edge_label = test_edge_label.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

print(model)

HeteroLinkPredictor(
  (embeddings): ModuleDict(
    (gene): Embedding(5292, 64)
    (drug): Embedding(18481, 64)
    (variant): Embedding(2826, 64)
    (disease): Embedding(756, 64)
  )
  (conv1): HeteroConv(num_relations=8)
  (conv2): HeteroConv(num_relations=8)
  (decoder): EdgeDecoder(
    (mlp): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=64, out_features=1, bias=True)
    )
  )
)


In [11]:
def precision_recall_at_k(y_true, y_score, k):
    order = np.argsort(-y_score)
    y_true_sorted = np.array(y_true)[order]
    topk = y_true_sorted[:k]

    precision_k = topk.mean() if len(topk) > 0 else 0.0
    total_positives = np.sum(y_true_sorted)
    recall_k = np.sum(topk) / total_positives if total_positives > 0 else 0.0
    return precision_k, recall_k


@torch.no_grad()
def evaluate_split(model, data, edge_label_index, edge_label):
    model.eval()
    z_dict = model(data)
    logits = model.decode(z_dict, edge_label_index)
    probs = torch.sigmoid(logits).cpu().numpy()
    y_true = edge_label.cpu().numpy()

    roc_auc = roc_auc_score(y_true, probs)
    ap = average_precision_score(y_true, probs)

    metrics = {
        "roc_auc": roc_auc,
        "average_precision": ap,
    }

    for k in [10, 25, 50, 100, 250, 500]:
        p_at_k, r_at_k = precision_recall_at_k(y_true, probs, k)
        metrics[f"precision@{k}"] = p_at_k
        metrics[f"recall@{k}"] = r_at_k

    return metrics, probs, y_true

In [16]:
EPOCHS = 500
best_val_ap = -1.0
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    z_dict = model(data)
    logits = model.decode(z_dict, train_edge_label_index)
    loss = criterion(logits, train_edge_label)

    loss.backward()
    optimizer.step()

    train_probs = torch.sigmoid(logits).detach().cpu().numpy()
    train_y = train_edge_label.detach().cpu().numpy()
    train_roc_auc = roc_auc_score(train_y, train_probs)
    train_ap = average_precision_score(train_y, train_probs)

    val_metrics, _, _ = evaluate_split(model, data, val_edge_label_index, val_edge_label)

    row = {
        "epoch": epoch,
        "train_loss": float(loss.item()),
        "train_roc_auc": float(train_roc_auc),
        "train_average_precision": float(train_ap),
        "val_roc_auc": float(val_metrics["roc_auc"]),
        "val_average_precision": float(val_metrics["average_precision"]),
    }
    history.append(row)

    if val_metrics["average_precision"] > best_val_ap:
        best_val_ap = val_metrics["average_precision"]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:03d} | "
            f"loss={loss.item():.4f} | "
            f"train AP={train_ap:.4f} | "
            f"val AP={val_metrics['average_precision']:.4f}"
        )

history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / "training_history.csv", index=False)
print("Best val AP:", best_val_ap)

Epoch 001 | loss=0.0150 | train AP=0.9999 | val AP=0.8984
Epoch 010 | loss=0.0141 | train AP=0.9999 | val AP=0.8998
Epoch 020 | loss=0.0150 | train AP=0.9999 | val AP=0.9001
Epoch 030 | loss=0.0144 | train AP=0.9999 | val AP=0.8980
Epoch 040 | loss=0.0158 | train AP=0.9999 | val AP=0.8990
Epoch 050 | loss=0.0149 | train AP=0.9999 | val AP=0.8985
Epoch 060 | loss=0.0129 | train AP=0.9999 | val AP=0.8982
Epoch 070 | loss=0.0126 | train AP=0.9999 | val AP=0.8988
Epoch 080 | loss=0.0126 | train AP=0.9999 | val AP=0.8986
Epoch 090 | loss=0.0123 | train AP=0.9999 | val AP=0.8992
Epoch 100 | loss=0.0121 | train AP=0.9999 | val AP=0.8990
Epoch 110 | loss=0.0119 | train AP=0.9999 | val AP=0.8987
Epoch 120 | loss=0.0115 | train AP=0.9999 | val AP=0.8998
Epoch 130 | loss=0.0117 | train AP=0.9999 | val AP=0.9006
Epoch 140 | loss=0.0113 | train AP=0.9999 | val AP=0.9009
Epoch 150 | loss=0.0115 | train AP=0.9999 | val AP=0.9014
Epoch 160 | loss=0.0114 | train AP=0.9999 | val AP=0.8999
Epoch 170 | lo

In [17]:
model.load_state_dict(best_state)

val_metrics, val_probs, val_y = evaluate_split(model, data, val_edge_label_index, val_edge_label)
test_metrics, test_probs, test_y = evaluate_split(model, data, test_edge_label_index, test_edge_label)

print("Validation metrics")
print("ROC-AUC:", round(val_metrics["roc_auc"], 4))
print("Average Precision:", round(val_metrics["average_precision"], 4))

print("\nTest metrics")
print("ROC-AUC:", round(test_metrics["roc_auc"], 4))
print("Average Precision:", round(test_metrics["average_precision"], 4))

Validation metrics
ROC-AUC: 0.911
Average Precision: 0.9054

Test metrics
ROC-AUC: 0.9079
Average Precision: 0.9051


In [18]:
metric_rows = []
for split_name, metrics in [("val", val_metrics), ("test", test_metrics)]:
    row = {"split": split_name}
    row.update(metrics)
    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(OUT_DIR / "gnn_metrics.csv", index=False)
display(metrics_df)

,split,roc_auc,average_precision,precision@10,recall@10,precision@25,recall@25,precision@50,recall@50,precision@100,recall@100,precision@250,recall@250,precision@500,recall@500
0,val,0.910997,0.905368,1.0,0.001159,1.0,0.002897,1.0,0.005794,0.98,0.011357,0.964,0.027929,0.978,0.056669
1,test,0.907890,0.905119,1.0,0.001131,1.0,0.002827,1.0,0.005655,0.98,0.011083,0.988,0.027935,0.976,0.055191


In [19]:
def make_scored_df(pos_df, neg_df, probs):
    df = pd.concat([
        pos_df.assign(label=1),
        neg_df.assign(label=0)
    ], ignore_index=True).copy()

    df["pred_prob"] = probs
    df["pred_label_0.5"] = (df["pred_prob"] >= 0.5).astype(int)
    return df

val_scored = make_scored_df(val_pos, val_neg, val_probs)
test_scored = make_scored_df(test_pos, test_neg, test_probs)

val_scored.to_csv(OUT_DIR / "val_scored_predictions.csv", index=False)
test_scored.to_csv(OUT_DIR / "test_scored_predictions.csv", index=False)

display(val_scored.head())

,source_id,source_raw,target_id,target_raw,split,label,pred_prob,pred_label_0.5
0,gene:nfe2l2,nfe2l2,drug:amlexanox,amlexanox,val,1,0.999951,1
1,gene:dnmt3a,dnmt3a,drug:galunisertib,galunisertib,val,1,0.999359,1
2,gene:cyp7b1,cyp7b1,drug:acetaminophen,acetaminophen,val,1,0.999715,1
3,gene:cyp2b6,cyp2b6,drug:methadone hydrochloride,methadone hydrochloride,val,1,0.999380,1
4,gene:cyp1a2,cyp1a2,drug:sulindac,sulindac,val,1,0.999946,1


In [20]:
trainval_edge_label_index = torch.cat([train_edge_label_index, val_edge_label_index], dim=1)
trainval_edge_label = torch.cat([train_edge_label, val_edge_label], dim=0)

final_model = HeteroLinkPredictor(
    metadata=data.metadata(),
    num_nodes_dict=num_nodes_dict,
    hidden_channels=64,
    out_channels=64
).to(device)

final_optimizer = torch.optim.Adam(final_model.parameters(), lr=0.005, weight_decay=1e-4)
final_criterion = nn.BCEWithLogitsLoss()

for epoch in range(1, 101):
    final_model.train()
    final_optimizer.zero_grad()

    z_dict = final_model(data)
    logits = final_model.decode(z_dict, trainval_edge_label_index)
    loss = final_criterion(logits, trainval_edge_label)

    loss.backward()
    final_optimizer.step()

    if epoch % 20 == 0 or epoch == 1:
        print(f"Final model epoch {epoch:03d} | loss={loss.item():.4f}")

Final model epoch 001 | loss=0.6944
Final model epoch 020 | loss=0.4375
Final model epoch 040 | loss=0.2893
Final model epoch 060 | loss=0.2215
Final model epoch 080 | loss=0.1862
Final model epoch 100 | loss=0.1618


In [21]:
known_positive_pairs = set(
    map(tuple,
        pd.concat([train_pos, val_pos, test_pos], ignore_index=True)[["source_id", "target_id"]].to_records(index=False)
    )
)

all_genes = nodes_by_type["gene"]
all_drugs = nodes_by_type["drug"]

candidate_pairs = []
for g in all_genes:
    for d in all_drugs:
        if (g, d) in known_positive_pairs:
            continue
        candidate_pairs.append((g, d))

candidate_df = pd.DataFrame(candidate_pairs, columns=["source_id", "target_id"])
print("candidate unseen pairs:", len(candidate_df))

candidate unseen pairs: 97732446


In [ ]:
@torch.no_grad()
def score_candidates_in_batches(model, data, candidate_df, batch_size=50000):
    model.eval()
    z_dict = model(data)

    scored_parts = []
    for start in range(0, len(candidate_df), batch_size):
        end = min(start + batch_size, len(candidate_df))
        batch = candidate_df.iloc[start:end].copy()

        src = torch.tensor([node2idx["gene"][x] for x in batch["source_id"]], dtype=torch.long, device=device)
        dst = torch.tensor([node2idx["drug"][x] for x in batch["target_id"]], dtype=torch.long, device=device)
        edge_label_index = torch.stack([src, dst], dim=0)

        logits = model.decode(z_dict, edge_label_index)
        probs = torch.sigmoid(logits).cpu().numpy()

        batch["pred_prob"] = probs
        scored_parts.append(batch)

        print(f"Scored {end:,} / {len(candidate_df):,}")

    return pd.concat(scored_parts, ignore_index=True)

candidate_scored = score_candidates_in_batches(final_model, data, candidate_df, batch_size=50000)
candidate_scored = candidate_scored.sort_values("pred_prob", ascending=False).reset_index(drop=True)

candidate_scored.to_csv(OUT_DIR / "all_scored_candidate_pairs.csv", index=False)
display(candidate_scored.head(20))

Scored 50,000 / 97,732,446
Scored 100,000 / 97,732,446
Scored 150,000 / 97,732,446
Scored 200,000 / 97,732,446
Scored 250,000 / 97,732,446
Scored 300,000 / 97,732,446
Scored 350,000 / 97,732,446
Scored 400,000 / 97,732,446
Scored 450,000 / 97,732,446
Scored 500,000 / 97,732,446
Scored 550,000 / 97,732,446
Scored 600,000 / 97,732,446
Scored 650,000 / 97,732,446
Scored 700,000 / 97,732,446
Scored 750,000 / 97,732,446
Scored 800,000 / 97,732,446
Scored 850,000 / 97,732,446
Scored 900,000 / 97,732,446
Scored 950,000 / 97,732,446
Scored 1,000,000 / 97,732,446
Scored 1,050,000 / 97,732,446
Scored 1,100,000 / 97,732,446
Scored 1,150,000 / 97,732,446
Scored 1,200,000 / 97,732,446
Scored 1,250,000 / 97,732,446
Scored 1,300,000 / 97,732,446
Scored 1,350,000 / 97,732,446
Scored 1,400,000 / 97,732,446
Scored 1,450,000 / 97,732,446
Scored 1,500,000 / 97,732,446
Scored 1,550,000 / 97,732,446
Scored 1,600,000 / 97,732,446
Scored 1,650,000 / 97,732,446
Scored 1,700,000 / 97,732,446
Scored 1,750,000 / 

In [ ]:
TOP_K_EXPORT = 500

top_candidates = candidate_scored.head(TOP_K_EXPORT).copy()
top_candidates.to_csv(OUT_DIR / "top_predicted_novel_gene_drug_links.csv", index=False)

print("saved:", OUT_DIR / "top_predicted_novel_gene_drug_links.csv")
display(top_candidates.head(50))

In [ ]:
experiment_summary = pd.DataFrame([
    {"item": "device", "value": str(device)},
    {"item": "num_gene_nodes", "value": len(nodes_by_type["gene"])},
    {"item": "num_drug_nodes", "value": len(nodes_by_type["drug"])},
    {"item": "num_variant_nodes", "value": len(nodes_by_type["variant"])},
    {"item": "num_disease_nodes", "value": len(nodes_by_type["disease"])},
    {"item": "num_edge_types_after_undirected", "value": len(data.edge_types)},
    {"item": "best_val_average_precision", "value": best_val_ap},
    {"item": "val_roc_auc", "value": val_metrics["roc_auc"]},
    {"item": "val_average_precision", "value": val_metrics["average_precision"]},
    {"item": "test_roc_auc", "value": test_metrics["roc_auc"]},
    {"item": "test_average_precision", "value": test_metrics["average_precision"]},
])

experiment_summary.to_csv(OUT_DIR / "experiment_summary.csv", index=False)
display(experiment_summary)